In [2]:
# import cadquery as cq

# # 단위는 보통 mm로 가정 (CadQuery 자체는 무차원이라 일관되게 쓰면 됨)
# A = 30.0  # overall height
# B = 25.0  # overall width
# C = 10.0  # stack depth (thickness)
# E = 18.0  # window height
# F = 12.0  # window width

# # 외형(실체)
# outer = cq.Workplane("XY").box(B, A, C, centered=(True, True, True))

# # 윈도우(빼낼 부피): 정중앙 관통 구멍
# window = cq.Workplane("XY").box(F, E, C + 1.0, centered=(True, True, True))

# core = outer.cut(window)

In [3]:
from __future__ import annotations

import os
from pathlib import Path

from pyaedt import Desktop, Maxwell3d
from typing import Iterable, Sequence, Any
from sympy import Expr


class MaxwellEddyCurrentSession:
    def __init__(self, project_path: Path, design_name: str) -> None:
        self.project_path = project_path
        self.design_name = design_name
        self.desktop = self._start_desktop()
        self.m3d = Maxwell3d(
            project=str(self.project_path),
            design=self.design_name,
            solution_type="EddyCurrent",
            non_graphical=True,
            new_desktop=False,
        )

    def _start_desktop(self) -> Desktop:
        version = os.getenv("AEDT_VERSION")
        if version:
            return Desktop(version=version, non_graphical=True, new_desktop=False)
        return Desktop(non_graphical=False, new_desktop=False)

    def create_box(
        self,
        origin: Sequence[Expr | float | int | str],
        sizes: Sequence[Expr | float | int | str],
        name: str | None = None,
    ) -> Any:
        def _as_str(values: Iterable[Expr | float | int | str]) -> list[str]:
            return [str(v) for v in values]
        
        from ansys.aedt.core.modeler.modeler_3d import Modeler3D
        modeler = self.m3d.modeler
        assert isinstance(modeler, Modeler3D)
        
        return modeler.create_box(origin=_as_str(origin), sizes=_as_str(sizes), name=name)

    def save(self) -> None:
        self.m3d.save_project()

    def close(self) -> None:
        self.desktop.close_desktop()


In [4]:
project_path = Path("/home/harry/Projects/AedtProjects/byPeetsFea").resolve() / "maxwell_eddy_current.aedt"
design_name = "EddyCurrentDesign"

session = MaxwellEddyCurrentSession(project_path, design_name)
session.save()
# session.desktop.release_desktop(close_on_exit=False, close_projects=False)


PyAEDT INFO: Python version 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:16:04) [GCC 11.2.0].
PyAEDT INFO: PyAEDT version 0.24.1.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file /tmp/pyaedt_harry_d3063ab1-35a7-49d4-a0d5-65a1adf6ec9b.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: New AEDT session is starting on gRPC port 44277.
PyAEDT INFO: Connecting to AEDT gRPC session on port 44277.
PyAEDT INFO: AEDT installation Path /opt/ansys_inc/v252/AnsysEM
PyAEDT INFO: Client application successfully started.
PyAEDT INFO: 2025.2 version started with process ID 1850760.
PyAEDT WARNING: Service Pack is not detected. PyAEDT is currently connecting in Insecure Mode.
PyAEDT WARNING: Please download and install latest Service Pack to use connect to AEDT in Secure Mode.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Python version 3.12.12 | packaged by Anaconda,

In [5]:
# UF-core class (TOML-driven, metaprogrammed from spec)
from __future__ import annotations

from dataclasses import dataclass
from typing import Mapping, Sequence, Any
import re
import tomllib
from sympy import symbols

_VALID_VAR_RE = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")

@dataclass
class UFCoreToml:
    session: Any
    spec_path: Path
    units: str | None = None
    dims: Mapping[str, float] | None = None

    def _load(self) -> tuple[str, Mapping[str, float]]:
        if self.units is not None and self.dims is not None:
            return self.units, self.dims
        with open(self.spec_path, "rb") as f:
            spec = tomllib.load(f)
        units = spec.get("meta", {}).get("units", "mm")
        dims: Mapping[str, float] = spec["dimensions"]
        required = ("A", "B", "C", "D", "E", "z_oversize")
        missing = [k for k in required if k not in dims]
        if missing:
            raise ValueError(f"Missing required dimensions in TOML: {missing}")
        self.units = units
        self.dims = dims
        return units, dims

    def _normalize_suffix(self, name_suffix: str) -> str:
        clean = re.sub(r"[^A-Za-z0-9_]", "_", name_suffix)
        if not clean or clean[0].isdigit():
            clean = f"s_{clean}"
        if not _VALID_VAR_RE.match(f"A_{clean}"):
            clean = "uf"
        return clean

    def _symbols(self, names: Sequence[str], name_suffix: str) -> list[Any]:
        sym_map = {name: symbols(f"{name}_{name_suffix}") for name in names}
        return [sym_map[name] for name in names]

    def build(self, name_suffix: str = "uf") -> Any:
        units, dims = self._load()
        suffix = self._normalize_suffix(name_suffix)
        modeler = self.session.m3d.modeler
        from ansys.aedt.core.modeler.modeler_3d import Modeler3D
        assert isinstance(modeler, Modeler3D)
        modeler.model_units = units

        for name, val in dims.items():
            self.session.m3d[f"{name}_{suffix}"] = f"{val}{units}"

        A, B, C, D, E, z_oversize = self._symbols(
            ["A", "B", "C", "D", "E", "z_oversize"],
            suffix,
        )
        F = B - D

        F_val = dims["B"] - dims["D"]
        if F_val <= 0:
            raise ValueError(
                f"Invalid dimensions: B({dims['B']}) must be > D({dims['D']}). Computed F={F_val}."
            )

        outer = self.session.create_box(
            origin=[-B / 2, -A / 2, -C / 2],
            sizes=[B, A, C],
            name=f"outer_{suffix}",
        )

        window = self.session.create_box(
            origin=[-B / 2, -E / 2, -(C + z_oversize) / 2],
            sizes=[F, E, C + z_oversize],
            name=f"window_{suffix}",
        )

        core = modeler.subtract(outer, [window], keep_originals=False)
        return core


In [ ]:
# Build UF-core from TOML using the class (run this instead of the manual cell)
uf_core = UFCoreToml(session=session, spec_path=Path("uf_core_d.toml"))
core = uf_core.build(name_suffix="class1")


PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec


In [1]:
from pyaedt import Desktop, Maxwell3d
# session.desktop.release_desktop(close_on_exit=False, close_projects=False)
Desktop(new_desktop=False).close_desktop()

PyAEDT INFO: Python version 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:16:04) [GCC 11.2.0].
PyAEDT INFO: PyAEDT version 0.24.1.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file /tmp/pyaedt_harry_d3063ab1-35a7-49d4-a0d5-65a1adf6ec9b.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Found active AEDT gRPC session on port 53179.
PyAEDT INFO: Connecting to AEDT gRPC session on port 53179.
PyAEDT INFO: AEDT installation Path /opt/ansys_inc/v252/AnsysEM
PyAEDT INFO: Client application successfully started.
PyAEDT WARNING: Service Pack is not detected. PyAEDT is currently connecting in Insecure Mode.
PyAEDT WARNING: Please download and install latest Service Pack to use connect to AEDT in Secure Mode.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Desktop has been released and closed.


True